In [1]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Load .env from project root
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

PROJECT_ID = os.getenv("PROJECT_ID")
assert PROJECT_ID, "PROJECT_ID not found in .env"

BASE_URL = f"https://firestore.googleapis.com/v1/projects/{PROJECT_ID}/databases/(default)/documents/recordings"
print(f"Project: {PROJECT_ID}")
print(f"Firestore URL: {BASE_URL}")

Project: data-collect-kateye
Firestore URL: https://firestore.googleapis.com/v1/projects/data-collect-kateye/databases/(default)/documents/recordings


In [2]:
def unwrap(val):
    """Unwrap a Firestore REST API typed value into a plain Python value.
    Mirrors the unwrap() logic in server.js:61-76."""
    if val is None:
        return None
    if "doubleValue" in val:
        return val["doubleValue"]
    if "integerValue" in val:
        return int(val["integerValue"])
    if "stringValue" in val:
        return val["stringValue"]
    if "booleanValue" in val:
        return val["booleanValue"]
    if "arrayValue" in val:
        return [unwrap(v) for v in val["arrayValue"].get("values", [])]
    if "mapValue" in val:
        return {k: unwrap(v) for k, v in val["mapValue"].get("fields", {}).items()}
    return None

In [3]:
resp = requests.get(BASE_URL)
resp.raise_for_status()
body = resp.json()
docs = body.get("documents", [])
print(f"Fetched {len(docs)} documents")

rows = []
for doc in docs:
    doc_id = doc["name"].split("/")[-1]
    fields = doc.get("fields", {})

    # Scalar fields
    row = {
        "id": doc_id,
        "label": unwrap(fields.get("label")),
        "label_name": unwrap(fields.get("label_name")),
        "device_id": unwrap(fields.get("device_id")),
        "sample_rate_hz": unwrap(fields.get("sample_rate_hz")),
        "num_samples": unwrap(fields.get("num_samples")),
        "duration_sec": unwrap(fields.get("duration_sec")),
        "accel_range_g": unwrap(fields.get("accel_range_g")),
        "gyro_range_dps": unwrap(fields.get("gyro_range_dps")),
        "dlpf_hz": unwrap(fields.get("dlpf_hz")),
        "gravity_axis": unwrap(fields.get("gravity_axis")),
        "gravity_sign": unwrap(fields.get("gravity_sign")),
        "flagged": unwrap(fields.get("flagged")),
        "notes": unwrap(fields.get("notes")),
        "timestamp": unwrap(fields.get("timestamp")),
    }

    # Calibration biases (nested mapValue)
    cal = unwrap(fields.get("calibration")) or {}
    for key in ["bias_ax", "bias_ay", "bias_az", "bias_gx", "bias_gy", "bias_gz"]:
        row[key] = cal.get(key)

    # Sensor data arrays (nested mapValue of arrayValues)
    data = unwrap(fields.get("data")) or {}
    for key in ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]:
        row[key] = data.get(key)

    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df.head()

Fetched 13 documents
DataFrame shape: (13, 27)


,id,label,label_name,device_id,sample_rate_hz,num_samples,duration_sec,accel_range_g,gyro_range_dps,dlpf_hz,...,bias_az,bias_gx,bias_gy,bias_gz,accel_x,accel_y,accel_z,gyro_x,gyro_y,gyro_z
0,2pJ3Qe2BbZQJUxH5PceK,8,Right,kateye-collector-01,56,560,10,8,500,21,...,47.72021,-189.32,-53.400,-38.660,"[-1.175176, -1.160806, -1.175176, -1.201521, -...","[0.182968, 0.182968, 0.173387, 0.156622, 0.156...","[9.731635, 9.73882, 9.669364, 9.774745, 9.7460...","[-0.003379, 0.00275, 0.002483, 0.001418, 0.003...","[0.001705, 0.01263, 0.005169, -0.007088, -0.01...","[0.000176, -0.00089, -0.001423, -0.002222, -9...."
1,6IqEHORQqigznbPxLPv2,6,Idling,kateye-collector-01,56,560,10,8,500,21,...,25.87012,-190.71,-55.045,-35.845,"[-0.03959, 0.017891, 0.039446, -0.006059, -0.0...","[0.032752, -0.017544, 0.004012, 0.013592, -0.0...","[9.834262, 9.841447, 9.776781, 9.795941, 9.879...","[-7.7e-05, 0.001522, 0.000189, -0.001143, -0.0...","[-0.000787, -0.001587, -0.000521, -0.000787, 1...","[-4.1e-05, 0.000225, 0.001557, -0.001374, -0.0..."
2,73Keb7j2o0pisnfjaHpM,6,Idling,kateye-collector-01,56,560,10,8,500,21,...,31.44482,-187.39,-56.700,-34.815,"[0.017304, 0.012514, 0.000539, 0.017304, 0.029...","[0.00194, -0.072306, -0.02201, -0.005245, -0.0...","[9.78738, 9.722714, 9.861626, 9.79217, 9.83528...","[0.000104, -0.000695, -0.001228, 0.000637, 0.0...","[-0.001679, -0.001146, 0.000453, -8e-05, -8e-0...","[0.001283, 0.001549, 0.002082, -0.00298, -0.00..."
3,GY6xFugNFg6j9nxmuIzW,6,Idling,kateye-collector-01,56,560,10,8,500,21,...,31.44482,-187.39,-56.700,-34.815,"[0.007724, -0.018621, -0.023411, 0.002934, 0.0...","[-0.00764, -0.01243, 0.02589, 0.0211, 0.009125...","[9.770615, 9.7778, 9.823305, 9.80654, 9.770615...","[-0.001228, -0.000962, 0.001436, -0.002294, 0....","[-0.001945, -0.000613, -0.000879, -0.002212, -...","[-0.000849, -0.001915, -0.001382, 0.000484, -0..."
4,NI0SSDVTUTMlbwHZstpz,4,Aggressive Right,kateye-collector-01,56,560,10,8,500,21,...,25.87012,-190.71,-55.045,-35.845,"[-1.134114, 0.178357, -0.343757, -0.319807, -0...","[1.203916, 1.266187, 1.522454, 0.703357, -0.11...","[10.16717, 9.599549, 10.19351, 9.755226, 9.563...","[-0.36673, -0.380319, -0.248154, -0.108527, 0....","[0.123917, 0.013069, 0.074089, -0.052481, -0.1...","[-0.155922, -0.086908, 0.054051, 0.179288, 0.2..."


In [4]:
# Class distribution
print("=== Class Distribution ===")
print(df.label_name.value_counts())

# Calibration stats
print("\n=== Calibration Bias Stats ===")
bias_cols = ["bias_ax", "bias_ay", "bias_az", "bias_gx", "bias_gy", "bias_gz"]
df[bias_cols].describe()

=== Class Distribution ===
label_name
Idling                   5
Aggressive Right         2
Right                    1
Left                     1
Aggressive Accelerate    1
Aggressive Left          1
Aggressive Brake         1
Brake                    1
Name: count, dtype: int64

=== Calibration Bias Stats ===


,bias_ax,bias_ay,bias_az,bias_gx,bias_gy,bias_gz
count,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000
mean,121.673077,8.735385,30.552658,-188.815385,-55.809615,-35.506923
std,121.413798,20.011739,5.844708,1.643789,1.090830,1.074891
min,-21.470000,-48.395000,25.870120,-190.710000,-56.700000,-38.660000
25%,-21.470000,1.325000,25.870120,-190.710000,-56.700000,-35.845000
50%,224.775000,22.190000,31.444820,-187.390000,-56.700000,-34.815000
75%,224.775000,22.190000,31.444820,-187.390000,-55.045000,-34.815000
max,224.775000,22.190000,47.720210,-187.390000,-53.400000,-34.815000


In [5]:
## Cell 5: Imports & Constants

import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "driving_events_model", "helpers"))

from build_dataset import normalize, build_datasets
from split_data import split_data
from build_cnn_model import compute_class_weights, save_model
from train_eval import train_model, evaluate_model
from plot_save_results import plot_results
from tflite_convert import convert_to_tflite, verify_tflite, export_c_header

import keras

# Constants
WINDOW_SIZE = 112
HOP_SIZE = 56
N_CHANNELS = 6
N_CLASSES = 9
BATCH_SIZE = 32
FINETUNE_LR = 1e-4
MAX_EPOCHS = 100
PATIENCE = 15

SENSOR_COLS = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]
BIAS_COLS = ["bias_ax", "bias_ay", "bias_az", "bias_gx", "bias_gy", "bias_gz"]
CLASS_NAMES = [
    "Accelerate",
    "Aggressive Accelerate",
    "Aggressive Brake",
    "Aggressive Left",
    "Aggressive Right",
    "Brake",
    "Idling",
    "Left",
    "Right",
]

PRETRAINED_MODEL_PATH = os.path.join(
    os.path.dirname(os.getcwd()), "model", "output", "driving_cnn.keras"
)
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Pretrained model: {PRETRAINED_MODEL_PATH}")
print(f"Exists: {os.path.exists(PRETRAINED_MODEL_PATH)}")

Pretrained model: /Users/tienle/Documents/Coding/KatEye/model/model/output/driving_cnn.keras
Exists: True


In [6]:
## Cell 6: Data Quality Filtering

# Drop flagged recordings
df_clean = df[df["flagged"] != True].copy()
print(f"After dropping flagged: {len(df_clean)} / {len(df)} recordings")

# Drop rows with missing sensor data
sensor_mask = df_clean[SENSOR_COLS].apply(lambda col: col.map(lambda v: v is not None and len(v) > 0))
df_clean = df_clean[sensor_mask.all(axis=1)].copy()
print(f"After dropping missing sensor data: {len(df_clean)} recordings")

# Drop recordings shorter than WINDOW_SIZE
df_clean["rec_len"] = df_clean[SENSOR_COLS[0]].apply(len)
df_clean = df_clean[df_clean["rec_len"] >= WINDOW_SIZE].copy()
print(f"After dropping short recordings (< {WINDOW_SIZE} samples): {len(df_clean)} recordings")

# Class distribution
print("\n=== Filtered Class Distribution ===")
print(df_clean["label_name"].value_counts())

After dropping flagged: 13 / 13 recordings
After dropping missing sensor data: 13 recordings
After dropping short recordings (< 112 samples): 13 recordings

=== Filtered Class Distribution ===
label_name
Idling                   5
Aggressive Right         2
Right                    1
Left                     1
Aggressive Accelerate    1
Aggressive Left          1
Aggressive Brake         1
Brake                    1
Name: count, dtype: int64


In [7]:
## Cell 7: Bias Correction + Window Extraction

def extract_windows_from_firestore(df_clean, sensor_cols, bias_cols, class_names, window_size, hop_size):
    """
    Extract sliding windows from Firestore wide-format recordings.
    Subtracts per-recording calibration biases before windowing.

    Returns:
        X_all     : np.ndarray (N, window_size, n_channels)
        y_all     : np.ndarray (N,)
        groups_all: np.ndarray (N,) — recording index per window (for stratified split)
    """
    activity_to_idx = {name: i for i, name in enumerate(class_names)}

    X_windows = []
    y_windows = []
    groups = []

    for rec_idx, (_, row) in enumerate(df_clean.iterrows()):
        label_name = row["label_name"]
        if label_name not in activity_to_idx:
            print(f"  Skipping unknown label: {label_name}")
            continue
        label = activity_to_idx[label_name]

        # Stack sensor channels into (num_samples, 6) array
        channels = []
        for s_col, b_col in zip(sensor_cols, bias_cols):
            raw = np.array(row[s_col], dtype=np.float32)
            bias = row[b_col] if row[b_col] is not None else 0.0
            channels.append(raw - bias)
        recording = np.stack(channels, axis=1)  # (num_samples, 6)

        # Sliding window
        num_samples = recording.shape[0]
        for start in range(0, num_samples - window_size + 1, hop_size):
            window = recording[start : start + window_size]
            X_windows.append(window)
            y_windows.append(label)
            groups.append(rec_idx)

    X_all = np.array(X_windows, dtype=np.float32)
    y_all = np.array(y_windows, dtype=np.int32)
    groups_all = np.array(groups, dtype=np.int32)
    return X_all, y_all, groups_all

X_all, y_all, groups_all = extract_windows_from_firestore(
    df_clean, SENSOR_COLS, BIAS_COLS, CLASS_NAMES, WINDOW_SIZE, HOP_SIZE
)

print(f"X_all shape: {X_all.shape}")  # (N, 112, 6)
print(f"y_all shape: {y_all.shape}")
print(f"Unique groups (recordings): {len(np.unique(groups_all))}")
print(f"\nWindows per class:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {np.sum(y_all == i)}")

X_all shape: (117, 112, 6)
y_all shape: (117,)
Unique groups (recordings): 13

Windows per class:
  Accelerate: 0
  Aggressive Accelerate: 9
  Aggressive Brake: 9
  Aggressive Left: 9
  Aggressive Right: 18
  Brake: 9
  Idling: 45
  Left: 9
  Right: 9


In [8]:
## Cell 8: Train/Val/Test Split

X_train, y_train, X_val, y_val, X_test, y_test = split_data(
    X_all, y_all, groups_all,
    class_names=CLASS_NAMES,
    n_splits=3,
    val_size=0.2,
    seed=42,
)

Train :     57 windows (48.7%)
Val   :     15 windows (12.8%)
Test  :     45 windows (38.5%)

Train class distribution:
  Accelerate: 0
  Aggressive Accelerate: 0
  Aggressive Brake: 7
  Aggressive Left: 7
  Aggressive Right: 7
  Brake: 7
  Idling: 22
  Left: 0
  Right: 7

Val class distribution:
  Accelerate: 0
  Aggressive Accelerate: 0
  Aggressive Brake: 2
  Aggressive Left: 2
  Aggressive Right: 2
  Brake: 2
  Idling: 5
  Left: 0
  Right: 2

Test class distribution:
  Accelerate: 0
  Aggressive Accelerate: 9
  Aggressive Brake: 0
  Aggressive Left: 0
  Aggressive Right: 9
  Brake: 0
  Idling: 18
  Left: 9
  Right: 0


In [9]:
## Cell 9: Normalize

X_train_norm, X_val_norm, X_test_norm, scaler = normalize(X_train, X_val, X_test)

# Save normalization params for C header export
train_mean = scaler.mean_
train_std = scaler.scale_
print(f"\ntrain_mean: {train_mean}")
print(f"train_std:  {train_std}")

Scaler fit on 57 training windows (6384 timesteps)
Channel means (train): [-150.6511   -8.1852  -22.2395  188.4595   55.9727   35.5925]
Channel stds  (train): [105.4994  23.2448   6.6184   1.602    1.2651   2.4365]
Post-norm train mean : 0.000001  (should be ≈ 0)
Post-norm train std  : 1.000000   (should be ≈ 1)

train_mean: [-150.65106718   -8.18520014  -22.23945025  188.45946315   55.9727332
   35.59245393]
train_std:  [105.49940777  23.24482286   6.6184417    1.60199705   1.26513186
   2.43650815]


In [10]:
## Cell 10: Build tf.data Pipelines

train_ds, val_ds, test_ds = build_datasets(
    X_train_norm, y_train,
    X_val_norm,   y_val,
    X_test_norm,  y_test,
    batch_size=BATCH_SIZE,
    n_channels=N_CHANNELS,
)

Train batch shape : X = (32, 112, 6), y = (32,)
Val   batch shape : X = (15, 112, 6), y = (15,)
Test  batch shape : X = (32, 112, 6), y = (32,)

Augmentation (train only): Gaussian noise (σ = 0.08) | scaling jitter [0.9, 1.1] | channel offset (σ = 0.02)


In [11]:
## Cell 11: Load Pretrained Model + Recompile

model = keras.models.load_model(PRETRAINED_MODEL_PATH)
print("Loaded pretrained model:")
model.summary()

# Recompile with lower learning rate for finetuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"\nRecompiled with Adam(lr={FINETUNE_LR})")

Loaded pretrained model:


Model: "DrivingCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sensor_input (InputLayer)       │ (None, 112, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 112, 16)        │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_1 (ReLU)                   │ (None, 112, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 112, 32)        │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_2 (ReLU)                   │ (None, 112, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool (MaxPooling1D)          │ (None, 56, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │        57,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_3 (ReLU)                   │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 9)              │           297 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178,637 (697.80 KB)

 Trainable params: 59,545 (232.60 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 119,092 (465.21 KB)


Recompiled with Adam(lr=0.0001)


In [12]:
## Cell 12: Compute Class Weights + Train

class_weight_dict = compute_class_weights(y_train, N_CLASSES)
print("Class weights:", class_weight_dict)

history = train_model(
    model,
    train_ds,
    val_ds,
    class_weight_dict,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)

  [Warning] Classes absent from training set (weight set to 1.0): {0, 1, 7}
Class weights: {0: 1.0, 1: 1.0, 2: 1.3571428571428572, 3: 1.3571428571428572, 4: 1.3571428571428572, 5: 1.3571428571428572, 6: 0.4318181818181818, 7: 1.0, 8: 1.3571428571428572}
Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.1754 - loss: 13.9976 - val_accuracy: 0.1333 - val_loss: 16.9982 - learning_rate: 1.0000e-04
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1404 - loss: 15.5655 - val_accuracy: 0.1333 - val_loss: 16.6992 - learning_rate: 1.0000e-04
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1579 - loss: 14.4528 - val_accuracy: 0.1333 - val_loss: 16.4106 - learning_rate: 1.0000e-04
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1754 - loss: 14.5594 - val_accuracy: 0.1333 - val_loss: 16.1293 - learning_rate: 1.0000e-04
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1228 - loss: 13.9355 - val_accuracy: 0.1333 - val_loss: 15.856

In [13]:
## Cell 13: Evaluate

y_pred, test_loss, test_acc = evaluate_model(
    model, test_ds, X_test_norm, y_test, CLASS_NAMES
)


Classification Report:
                       precision    recall  f1-score   support

           Accelerate     0.0000    0.0000    0.0000         0
Aggressive Accelerate     0.0000    0.0000    0.0000         9
     Aggressive Brake     0.0000    0.0000    0.0000         0
      Aggressive Left     0.0000    0.0000    0.0000         0
     Aggressive Right     0.1818    0.4444    0.2581         9
                Brake     0.0000    0.0000    0.0000         0
               Idling     0.0000    0.0000    0.0000        18
                 Left     0.4375    0.7778    0.5600         9
                Right     0.0000    0.0000    0.0000         0

             accuracy                         0.2444        45
            macro avg     0.0688    0.1358    0.0909        45
         weighted avg     0.1239    0.2444    0.1636        45

Test Loss     : 5.3087
Test Accuracy : 0.2444

Per-class accuracy:
  Accelerate: no samples in test set
  Aggressive Accelerate: 0.0000  (9 samples)
  Agg

/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])

In [14]:
## Cell 14: Plot Results

plot_results(
    history, y_test, y_pred, CLASS_NAMES,
    output_dir=OUTPUT_DIR,
    filename="finetune_results.png",
)

Saved figure → output/finetune_results.png


'output/finetune_results.png'

In [15]:
## Cell 15: Save Finetuned Keras Model

save_path = save_model(model, OUTPUT_DIR, filename="driving_cnn_finetuned.keras")
print(f"Saved finetuned model: {save_path}")

Saved Keras model to output/driving_cnn_finetuned.keras
File size: 746.6 KB
Saved finetuned model: output/driving_cnn_finetuned.keras


In [16]:
## Cell 16: TFLite INT8 Conversion + Verification

f32_path, int8_path, tflite_int8 = convert_to_tflite(
    model, X_train_norm, OUTPUT_DIR, model_name="driving_cnn_finetuned"
)

tflite_preds, tflite_acc, input_scale, input_zp = verify_tflite(
    int8_path, X_test_norm, y_test, test_acc
)

INFO:tensorflow:Assets written to: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e/assets


INFO:tensorflow:Assets written to: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e/assets


Saved artifact at '/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 112, 6), dtype=tf.float32, name='sensor_input')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  13645500048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645499664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645501776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645501968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
Float32 TFLite : 238.6 KB → output/driving_cnn_finetuned_f32.tflite


W0000 00:00:1776882831.225211 6547540 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776882831.225219 6547540 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776882831.225504 6547540 reader.cc:83] Reading SavedModel from: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e
I0000 00:00:1776882831.225718 6547540 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776882831.225720 6547540 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e
I0000 00:00:1776882831.227604 6547540 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1776882831.227935 6547540 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776882831.239923 6547540 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpb8hxll3e
I0000 00:00:1776882831.243821 6547540 lo

INFO:tensorflow:Assets written to: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpn9fi7tyr/assets


INFO:tensorflow:Assets written to: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpn9fi7tyr/assets


Saved artifact at '/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpn9fi7tyr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 112, 6), dtype=tf.float32, name='sensor_input')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  13645500048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645499664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645501776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645501968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645500624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13645502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
INT8 TFLite    : 67.1 KB → output/driving_cnn_finetuned_int8.tflite
Size reduction : 238.6 KB → 67.1 KB (71.9% smaller)
TFLite input  : [  1 112   6]

/Users/tienle/anaconda3/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1776882831.543553 6547540 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776882831.543560 6547540 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776882831.543626 6547540 reader.cc:83] Reading SavedModel from: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpn9fi7tyr
I0000 00:00:1776882831.543831 6547540 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776882831.543833 6547540 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/tmpn9fi7tyr
I0000 00:00:1776882831.545885 6547540 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776882831.557525 6547540 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folde

In [17]:
## Cell 17: C Header Export

header_path = export_c_header(
    tflite_int8=tflite_int8,
    output_dir=OUTPUT_DIR,
    input_scale=input_scale,
    input_zp=input_zp,
    window_size=WINDOW_SIZE,
    n_channels=N_CHANNELS,
    n_classes=N_CLASSES,
    sensor_cols=SENSOR_COLS,
    class_names=CLASS_NAMES,
    train_mean=train_mean,
    train_std=train_std,
    array_name="driving_cnn_finetuned_model",
    filename="driving_cnn_finetuned_model.h",
)
print(f"\nC header ready for ESP32-S3 firmware: {header_path}")

Saved C header → output/driving_cnn_finetuned_model.h
File size      : 420.6 KB
Model bytes    : 68,744

C header ready for ESP32-S3 firmware: output/driving_cnn_finetuned_model.h


In [20]:
y_pred, test_loss, test_acc = evaluate_model(
    model, test_ds, X_test_norm, y_test, CLASS_NAMES
)


Classification Report:
                       precision    recall  f1-score   support

           Accelerate     0.0000    0.0000    0.0000         0
Aggressive Accelerate     0.0000    0.0000    0.0000         9
     Aggressive Brake     0.0000    0.0000    0.0000         0
      Aggressive Left     0.0000    0.0000    0.0000         0
     Aggressive Right     0.1818    0.4444    0.2581         9
                Brake     0.0000    0.0000    0.0000         0
               Idling     0.0000    0.0000    0.0000        18
                 Left     0.4375    0.7778    0.5600         9
                Right     0.0000    0.0000    0.0000         0

             accuracy                         0.2444        45
            macro avg     0.0688    0.1358    0.0909        45
         weighted avg     0.1239    0.2444    0.1636        45

Test Loss     : 5.3087
Test Accuracy : 0.2444

Per-class accuracy:
  Accelerate: no samples in test set
  Aggressive Accelerate: 0.0000  (9 samples)
  Agg

/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tienle/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])